In [98]:
import pandas as pd
from pydantic_ai import Agent
import os
from pydantic_ai.models.huggingface import HuggingFaceModel
from pydantic_ai.providers.huggingface import HuggingFaceProvider
from pathlib import Path

os.environ["OLLAMA_BASE_URL"] = "http://localhost:11434/v1"

In [96]:
df = pd.read_csv("../preprocessed_data/preprocessed_data.csv")

In [97]:
df

,author,post,political_leaning,non_standard_word_ratio,most_common_word_ratio
0,t2_7ramzeng,"You can ""buy"" the show and stream it through t...",right,0.206667,0.033333
1,t2_7ramzeng,"me want to play Qbert Holy shit, based Alex Jo...",right,0.162667,0.044000
2,t2_7ramzeng,Shouldn't rely on any external services or per...,right,0.157438,0.044696
3,t2_7ramzeng,PR to a specific person. Usually that just mea...,right,0.149333,0.042000
4,t2_7ramzeng,This article's intention is clear that they wa...,right,0.152000,0.034667
...,...,...,...,...,...
56271,t2_4ngvl16j,a good one? That's odd. I remember it as being...,center,0.228667,0.031333
56272,t2_4ngvl16j,"boring shit in the fucking world. ""History doe...",center,0.230000,0.027333
56273,t2_4ngvl16j,you see no contradiction there? Why or why not...,center,0.195333,0.031333
56274,t2_4ngvl16j,is only created by an incommensurate worldview...,center,0.215333,0.035333


In [12]:
post_text = df.iloc[0].to_dict()["post"]

In [4]:
from typing import Literal
from pydantic import BaseModel, Field


class Judgment(BaseModel):
    is_leaky: bool = Field(..., description="Whether the post contains data leakage")
    severity_score: float = Field(
        ..., ge=0.0, le=1.0, description="Severity score from 0.0 to 1.0"
    )
    confidence: float = Field(
        ..., ge=0.0, le=1.0, description="Confidence in the judgment from 0.0 to 1.0"
    )


class Classification(BaseModel):
    leakage_domain: Literal["Semantic", "Orthographic", "Syntactic", "Structural"] = (
        Field(..., description="The domain of the leakage")
    )
    specific_mechanism: str = Field(
        ..., description="The specific mechanism of the leakage"
    )
    is_novel_category: bool = Field(
        ..., description="Whether the mechanism is a novel category"
    )
    definition: str = Field(..., description="Short description of the mechanism")


class Forensics(BaseModel):
    evidence_spans: list[str] = Field(
        ..., description="List of exact strings that constitute evidence"
    )
    evidence_location: Literal["Beginning", "Middle", "End", "Scattered", "N/A"] = (
        Field(..., description="Location of the evidence in the text")
    )
    pattern_abstraction: str = Field(
        ..., description="Technical description of the pattern logic"
    )


class JudgeResult(BaseModel):
    judgment: Judgment = Field(..., description="The overall judgment")
    classification: Classification | None = Field(
        None,
        description="Classification of the leakage (only present if is_leaky=true)",
    )
    forensics: Forensics | None = Field(
        None,
        description="Forensic analysis of the leakage (only present if is_leaky=true)",
    )

In [5]:
prompt_content = Path("../prompts/llmaj_prompt_by_grazie").read_text(encoding="utf-8")

In [6]:
# 2. Pass it to 'system_prompt'
model = HuggingFaceModel(
    "Qwen/Qwen3-235B-A22B-Instruct-2507",
    provider=HuggingFaceProvider(
        api_key=os.environ["HF_TOKEN"], provider_name="together"
    ),
)
# Qwen/Qwen3-235B-A22B-Instruct-2507:together - hf
# cerebras:qwen-3-235b-a22b-instruct-2507


# openai/gpt-oss-120b:novita
# meta-llama/Llama-3.3-70B-Instruct:novita
# Qwen/Qwen3-Next-80B-A3B-Instruct:hyperbolic
# "ollama:llama3.1:8b",
# 'groq:llama-3.3-70b-versatile',
# "ollama:huggingface.co/unsloth/Nemotron-3-Nano-30B-A3B-GGUF:latest",

In [7]:
import logfire

logfire.configure()
logfire.instrument_pydantic_ai()

Logfire project URL: ]8;id=631221;https://logfire-eu.pydantic.dev/skyrimer/lang-and-ai\https://logfire-eu.pydantic.dev/skyrimer/lang-and-ai]8;;\

In [8]:
agent = Agent(
    model, output_type=JudgeResult, system_prompt=prompt_content, output_retries=1
)

In [9]:
result = await agent.run(post_text)

12:26:19.429 agent run
12:26:19.434   chat Qwen/Qwen3-235B-A22B-Instruct-2507


In [17]:
result.output.judgment

Judgment(is_leaky=True, severity_score=0.75, confidence=0.92)

In [14]:
print(post_text)

You can "buy" the show and stream it through them. Including the Lethal Weapon 6. On Fridays we do a zoom meeting where we go over every ticket for the sprint and talk about its status. That tends to be more productive. I usually tell them. I had it sort of bite me in one interview though. It was like 2016 and the interviewers didn't even know Glassdoor was a thing, somehow. I checked before the interview to see what to expect, then they ended up asking me the exact same questions. I was honest, like "yeah, I've seen all these", then talked about my interview prep process and assured them I wasn't just rote memorizing answers to their questions, but that practice-interviewing myself with the questions on Glassdoor was the most effective way to prepare for the assumption that they'd ask similar but different questions. I could tell they were torn between thinking I had cheated but knowing I wouldn't be so bluntly honest about it if I had. Like seriously guys, your last several hires hav

# Df test

In [17]:
df[df["is_similarity_clustered"]]

,author,post,political_leaning,similarity_cluster_id,is_similarity_clustered
56642,t2_um1d98xf,don't see your flair >:( *** ^(User hasn't fla...,center,3361,True
56643,t2_p44a4,senior National Security Council official said...,right,9782,True
56644,t2_hrqan,worldwide** following PETA’s hard-hitting camp...,left,16684,True
56645,t2_hrqan,in violation of the 13th Amendment to the U.S....,left,16685,True
56646,t2_hrqan,"Washington, D.C. · **1981** : PETA conducts an...",left,16721,True
56647,t2_pkpac1sb,he is the messiah of idiots. Allah forbids you...,right,33694,True
56648,t2_fdw7e,"centre of things, because they are sure real p...",center,34582,True
56649,t2_9ixpncmr,вуглеводів і не забувайте їсти та зволожувати....,center,34693,True
56650,t2_3vumojbw,cum cum cum cumcum cum cum cum cum cum cum cum...,right,46027,True
56651,t2_5wzio,in prison. url Republican congressman Mark Fol...,left,47971,True


In [47]:
df["post_length"] = df["post"].str.split().str.len()
df["post_length_chr"] = df["post"].str.len()

In [49]:
df[df["post_length_chr"] > 10_000]

,author,post,political_leaning,post_length,post_length_chr
289,t2_yhe63e4,that bad anymore Metal music. Cannibal Corpse ...,center,1500,12966
2121,t2_6edxz2zq,which lead to the sub being in fact nothing bu...,center,1500,10121
2128,t2_6edxz2zq,since there are well-developed environmental a...,center,1500,10272
2129,t2_6edxz2zq,just a straight up clip from a show? Flair up ...,center,1500,10134
3144,t2_1imyap,"głosować. Nie mówię, że jesteś Marksistą. Tylk...",right,1500,10215
...,...,...,...,...,...
55329,t2_2567bbu4,though Edit or just frenzy for single target D...,right,1500,10827
55330,t2_2567bbu4,that the building must comply with including h...,right,1496,10180
55331,t2_2567bbu4,samassa ajassa (~30-40v) takaisin verrattuna t...,right,1500,10455
56282,t2_3myjqrij,Tässä tullaan siihen homman ytimeen. Jos oikea...,left,1500,10345


In [91]:
from collections import Counter
import re


# Function to calculate the ratio of most common word
def most_common_word_ratio(text):
    words = text.split()
    if len(words) == 0:
        return 0.0
    word_counts = Counter(words)
    most_common_count = word_counts.most_common(1)[0][1]
    return most_common_count / len(words)


# Function to calculate the ratio of non-standard words
def non_standard_word_ratio(text):
    # Define pattern for standard words (letters, numbers, basic punctuation)

    words = text.split()

    non_standard_count = 0
    for word in words:
        if not word.isalnum():
            non_standard_count += 1

    return non_standard_count / len(words)


# Function to filter dataframe by non_standard_word_ratio threshold
def filter_by_non_standard_ratio(df, threshold=0.4):
    """
    Filter dataframe for posts that have non_standard_word_ratio less than threshold.

    Args:
        df: Input dataframe with 'post' column
        threshold: Maximum allowed non_standard_word_ratio (default: 0.4)

    Returns:
        Filtered dataframe
    """
    df_filtered = df.copy()
    if "non_standard_word_ratio" not in df_filtered.columns:
        df_filtered["non_standard_word_ratio"] = df_filtered["post"].apply(
            non_standard_word_ratio
        )

    return df_filtered[df_filtered["non_standard_word_ratio"] < threshold]


# Function to remove most common word if ratio exceeds threshold
def remove_common_word_by_ratio(df, threshold=0.3):
    """
    Compute most_common_word_ratio and remove the most common word from posts
    where the ratio exceeds the threshold.

    Args:
        df: Input dataframe with 'post' column
        threshold: Minimum ratio to trigger removal (default: 0.3)

    Returns:
        Dataframe with cleaned posts
    """
    df_cleaned = df.copy()

    # Compute most_common_word_ratio if not already present
    if "most_common_word_ratio" not in df_cleaned.columns:
        df_cleaned["most_common_word_ratio"] = df_cleaned["post"].apply(
            most_common_word_ratio
        )

    # Remove most common word where ratio exceeds threshold
    def sanitize_text(row):
        if row["most_common_word_ratio"] > threshold:
            words = row["post"].split()
            if len(words) == 0:
                return row["post"]

            word_counts = Counter(words)
            most_common_word = word_counts.most_common(1)[0][0]
            sanitized_words = [word for word in words if word != most_common_word]
            return " ".join(sanitized_words)
        return row["post"]

    df_cleaned["post"] = df_cleaned.apply(sanitize_text, axis=1)

    return df_cleaned


# Create the new columns
df["most_common_word_ratio"] = df["post"].apply(most_common_word_ratio)
df["non_standard_word_ratio"] = df["post"].apply(non_standard_word_ratio)

# Apply sanitization
# df['post'] = df['post'].apply(sanitize_by_most_common_word)

In [95]:
df[df["non_standard_word_ratio"] > 0.4]
# df["non_standard_word_ratio"]

,author,post,political_leaning,most_common_word_ratio,non_standard_word_ratio
3145,t2_1imyap,i anrachiści się zwalczali. W Rosji terytori...,right,0.021333,0.400667
3361,t2_um1d98xf,or others might bully you for the rest of your...,center,0.057263,0.453911
3362,t2_um1d98xf,other people >:) * ^(User hasn't flaired up ye...,center,0.047453,0.441731
3364,t2_um1d98xf,/ 197 ^^|| [[[Guide]]](url > Get a flair to ma...,center,0.043780,0.416956
3365,t2_um1d98xf,friendly reminder to HAVE YOUR FRICKIN' FLAIR ...,center,0.045993,0.432056
...,...,...,...,...,...
53508,t2_xse8j,nữa là còn chưa bao giờ nghe đến. Thu...,left,0.025367,0.606809
53509,t2_xse8j,"sang đây chơi thì cách ly làm l gì"". Cã...",left,0.027554,0.432796
53510,t2_xse8j,above is the simplistic version. How can you b...,left,0.021333,0.450000
53511,t2_xse8j,for AuthRight. Yeah I agree with you here. Ass...,left,0.034023,0.400267


<>:1: SyntaxWarning: invalid escape sequence '\-'
<>:1: SyntaxWarning: invalid escape sequence '\-'
/var/folders/fn/0zw43mf17dlbcmw9ypr0b1jh0000gn/T/ipykernel_87638/1571037925.py:1: SyntaxWarning: invalid escape sequence '\-'
  text = """protest and plans to take it to court ​ - SpaceX actually makes progress, Blue Origin can't even produce engines for New Glenn; Blue Origin continues to flounder as it has for 2 decades ​ - Salty informercials from Blue Origin Hope it helps. Edit: fixed first bullet point That it could certainly do. "Do not respond, *no matter how human they may seem.*" \- SCP EAS I was up at 3AM - the server collapse was very unpog I'd say it's more alike emission. Stretching the definition could include "fission", but I think that's 0.5 steps too far. It's over 1 magnitude (\~5x10\^1 times) more efficient, so I have that going for me. Amogus AuthLeft points to a study showing corporatism is inevitable AuthRight finally getting triggered for once LibLeft not being tri

'\'protest and plans to take it to court \\u200b - SpaceX actually makes progress, Blue Origin can\\\'t even produce engines for New Glenn; Blue Origin continues to flounder as it has for 2 decades \\u200b - Salty informercials from Blue Origin Hope it helps. Edit: fixed first bullet point That it could certainly do. "Do not respond, *no matter how human they may seem.*" \\\\- SCP EAS I was up at 3AM - the server collapse was very unpog I\\\'d say it\\\'s more alike emission. Stretching the definition could include "fission", but I think that\\\'s 0.5 steps too far. It\\\'s over 1 magnitude (\\\\~5x10\\\\^1 times) more efficient, so I have that going for me. Amogus AuthLeft points to a study showing corporatism is inevitable AuthRight finally getting triggered for once LibLeft not being triggered for once LibRight chokes a centrist for not wanting to sell their grill I do the science !img OP actually takes the titles of news seriously? Word. The screen isn\\\'t near big enough to be se

In [65]:
corpus_word_counter.most_common(101)

[('the', 3013682),
 ('to', 2051998),
 ('a', 1898756),
 ('and', 1581728),
 ('of', 1491814),
 ('is', 1206425),
 ('I', 1203026),
 ('in', 1016899),
 ('that', 1004498),
 ('you', 892946),
 ('it', 753807),
 ('for', 703280),
 ('are', 574041),
 ('be', 525082),
 ('not', 501006),
 ('have', 494510),
 ('with', 483397),
 ('on', 473677),
 ('but', 461921),
 ('they', 430258),
 ('as', 420448),
 ('was', 412759),
 ('just', 367170),
 ('this', 351273),
 ('like', 346945),
 ('or', 332202),
 ('The', 295641),
 ('if', 294549),
 ('at', 284448),
 ('your', 283157),
 ('people', 281687),
 ('can', 278446),
 ('an', 257343),
 ('so', 256070),
 ('about', 251309),
 ('my', 247750),
 ('from', 245542),
 ('would', 244286),
 ('more', 233261),
 ('do', 232403),
 ('all', 224928),
 ('their', 218745),
 ('what', 215594),
 ('get', 204998),
 ('one', 199588),
 ('by', 197108),
 ('because', 194205),
 ('he', 192026),
 ('think', 187865),
 ('we', 183609),
 ('has', 179828),
 ("don't", 174010),
 ('no', 173686),
 ('than', 173281),
 ('up', 17282

In [35]:
df_original["post_length"] = df_original["post"].str.split().str.len()

In [41]:
df_original[df_original["post_length"] < 300]

,auhtor_ID,post,political_leaning,post_length
2997,t2_3tdu1mrp,"""national propaganda radio"" comment, I listene...",right,295
5711,t2_715ysxax,&gt;Community demands mutilation to stay in. I...,left,271
6751,t2_3jbn6qmg,"to. Well yes, the British eat blood pudding an...",left,270
6880,t2_75lui9um,"Didn’t think about it like that, based Nah, I’...",left,290
7020,t2_ibpcxkta,be actual criteria for being trans. Having gen...,center,277
7167,t2_5vf1jq8p,pretty durable. Mine are my daily wear and at ...,center,288
10332,t2_c7ckm,"up somewhere, we're gonna start over from the ...",right,299
13621,t2_4y53h,and do for other things but paying double the ...,right,281
17039,t2_3scnnupg,ever seen Sure It seems today That all you see...,center,290
19201,t2_qlj9gk08,You can find some headlight polish cream at mo...,center,295


In [40]:
df_original["post_length"]

0        1500
1        1500
2        1500
3        1500
4        1500
         ... 
57226    1500
57227    1500
57228    1500
57229    1500
57230    1500
Name: post_length, Length: 57231, dtype: int64